# 01 — Clean DHS adult education data
**Education Under Fire: Conflict Exposure and Educational Attainment in Nigeria**

Builds an adult-level dataset of completed schooling from the 2003 and 2008 Nigeria DHS.

The Household Member Recode (PR) lists every member of every sampled household, with age, sex,
completed years of schooling and state of residence. Adults born 1935–1975 were school-age before,
during or after the 1967–70 civil war, and their schooling is complete by 2003/2008, so it measures
final educational attainment.

- **Input:** `data/raw/dhs/NG_2003_DHS/.../NGPR4CFL.DTA`, `data/raw/dhs/NG_2008_DHS/.../NGPR53FL.DTA`
- **Output:** `data/processed/dhs_adults.csv`

In [ ]:
import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt

from config import *

## 1. Load the PR file for each wave

In [ ]:
PR_VARS = {
    "hv001": "cluster",
    "hv002": "household",
    "hvidx": "line",
    "hv005": "hv005",
    "hv007": "interview_year",
    "hv103": "hv103",
    "hv104": "hv104",
    "hv105": "age",
    "hv108": "hv108",
    "hv109": "hv109",
    "hv025": "hv025",
    "shstate": "shstate",
}

waves = []
for survey_year, path in DHS_PR_FILES.items():
    df, meta = pyreadstat.read_dta(path, usecols=list(PR_VARS))
    df = df.rename(columns=PR_VARS)
    df["state_dhs"] = df["shstate"].map(meta.variable_value_labels["shstate"])
    df["survey_year"] = survey_year
    waves.append(df.drop(columns="shstate"))
    print(f"{survey_year}: {len(df):>7,} household members")

dhs = pd.concat(waves, ignore_index=True)

## 2. Harmonise state names

In [ ]:
def clean_state(name: str) -> str:
    key = name.strip().lower()
    return DHS_STATE_FIXES.get(key, key.title())

dhs["state"] = dhs["state_dhs"].map(clean_state)
print(f"{dhs['state'].nunique()} states")

## 3. Birth year and sample
DHS records age, not year of birth, for household members. Birth year is the interview year minus
age, so it can be off by one year. Ages 95+ are top-coded or unknown and are dropped. Only de facto
members (slept in the household the night before) are kept, the standard DHS population.

In [ ]:
dhs = dhs[(dhs["hv103"] == 1) & (dhs["age"] < 95)].copy()
dhs["birth_year"] = dhs["interview_year"] - dhs["age"]
dhs = dhs[dhs["birth_year"].between(*BIRTH_YEARS)]
print(f"Adults born {BIRTH_YEARS[0]}–{BIRTH_YEARS[1]}: {len(dhs):,}")
dhs.groupby("survey_year").size()

## 4. Age heaping
Older respondents often report ages rounded to a multiple of 5. Without heaping, about 20% of ages
would end in 0 or 5. Heaped ages blur birth years, so they are flagged for a robustness check.

In [ ]:
dhs["age_heaped"] = (dhs["age"] % 5 == 0).astype(int)
print(f"Share of ages ending in 0 or 5: {dhs['age_heaped'].mean():.1%} (about 20% expected without heaping)")
dhs.groupby(dhs["age"] % 10).size().plot(kind="bar", title="Last digit of reported age", figsize=(7, 3));

## 5. Outcomes and controls
- `years_educ`: completed years of schooling (`hv108`); codes 97–99 become NaN
- `completed_primary`: completed primary school or more (`hv109` ≥ 2); code 8 (don't know) becomes NaN

Current urban residence and household wealth are measured decades after schooling ended and could
themselves be affected by the war, so they are not used as controls.

In [ ]:
dhs["years_educ"] = dhs["hv108"].where(dhs["hv108"] < 90)
dhs["completed_primary"] = (dhs["hv109"] >= 2).astype(float).where(dhs["hv109"] != 8)
dhs["male"] = (dhs["hv104"] == 1).astype(int)
dhs["weight"] = dhs["hv005"] / 1e6

dhs["cluster_id"] = dhs["survey_year"].astype(str) + "_" + dhs["cluster"].astype(int).astype(str)
dhs["person_id"] = (dhs["cluster_id"] + "_" + dhs["household"].astype(int).astype(str)
                    + "_" + dhs["line"].astype(int).astype(str))
assert dhs["person_id"].is_unique

print("Share missing:")
print(dhs[["years_educ", "completed_primary"]].isna().mean().round(4))
dhs = dhs.dropna(subset=["years_educ", "completed_primary"])

## 6. First look: schooling by birth cohort

In [ ]:
dhs["cohort5"] = dhs["birth_year"] // 5 * 5
war_region = dhs["state"].isin(WAR_STATES).map({True: "War region (south-east)", False: "Rest of Nigeria"})

profile = dhs.groupby(["cohort5", war_region]).apply(
    lambda d: np.average(d["years_educ"], weights=d["weight"])
).unstack()

ax = profile.plot(marker="o", figsize=(9, 5))
ax.axvspan(1950, 1963, color="grey", alpha=0.15, label="School-age during the war (born 1950–63)")
ax.set_xlabel("Birth cohort (5-year bins)")
ax.set_ylabel("Years of schooling (weighted)")
ax.set_title("Completed schooling by birth cohort")
ax.legend(frameon=False)
plt.tight_layout()
profile.round(2)

## 7. Save

In [ ]:
keep = [
    "person_id", "cluster_id", "survey_year", "interview_year", "state", "weight",
    "age", "age_heaped", "birth_year", "male", "years_educ", "completed_primary",
]
dhs[keep].to_csv(DHS_ADULTS, index=False)
print(f"Saved {len(dhs):,} adults to {DHS_ADULTS.relative_to(PROJECT_ROOT)}")